# 3. Inference Pipeline

**Purpose**: Apply trained model at scale to 11M+ records via chunked processing.

## Sections
1. Load config and saved model
2. Define chunking strategy
3. Process chunks with progress tracking
4. Cluster predictions
5. Export results


---
## 1. Setup and Load Model


In [1]:
import sys
sys.path.insert(0, '/Users/robertlalani/Desktop/entity_resolution_12_18_25/01-05-26')

import pandas as pd
import numpy as np
import json
from pathlib import Path

# Splink imports
from splink import Linker, DuckDBAPI

# Local imports
from config import config
from utils import (
    DatabaseManager, 
    log_step, 
    Timer,
    ProgressTracker,
    save_checkpoint,
    load_checkpoint,
    load_json
)
from data_prep import (
    load_mismatched,
    load_dim_org,
    create_unified_schema,
    filter_bad_records,
    create_blocking_keys,
    add_distinctive_tokens,  # Changed from add_idf_based_features
    add_token_set_features,
    compute_token_statistics,
    get_corpus_stopwords
)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)

print("Imports loaded successfully")


Imports loaded successfully


In [2]:
# Initialize database
db = DatabaseManager()

# Load saved model
model_path = config.paths.MODEL_FILE
if not Path(model_path).exists():
    raise FileNotFoundError(f"Model not found at {model_path}. Run 2_training.ipynb first.")

model_json = load_json(model_path, "Trained model")

print(f"\nMODEL LOADED")
print("=" * 50)
print(f"Path: {model_path}")
if 'training_metadata' in model_json:
    meta = model_json['training_metadata']
    print(f"Trained on: {meta.get('timestamp', 'N/A')}")
    print(f"Training records: {meta.get('dim_org_records', 0):,} dim_org + {meta.get('grid_records', 0):,} GRID")


[15:23:17]  Loading JSON: Trained model

MODEL LOADED
Path: /Users/robertlalani/Desktop/entity_resolution_12_18_25/01-05-26/models/model_v1.json
Trained on: 2026-01-09T14:50:37.956908
Training records: 109,851 dim_org + 109,856 GRID


In [3]:
# Load reference data (dim_org) for linking
# Using cached data if available
cached_dim_org = load_checkpoint(config.paths.DATA_DIR + "/dim_org_training.parquet", "dim_org cache")

if cached_dim_org is not None:
    dim_org_df = cached_dim_org
    print(f"Loaded cached dim_org: {len(dim_org_df):,} records")
else:
    dim_org_raw = load_dim_org(db, sample_size=None)  # Full load for inference
    dim_org_df = create_unified_schema(dim_org_raw, 'dim_org')
    dim_org_df, _ = filter_bad_records(dim_org_df)
    dim_org_df = create_blocking_keys(dim_org_df)
    print(f"Loaded dim_org: {len(dim_org_df):,} records")


[15:23:17]  Loading checkpoint: dim_org cache
[15:23:17]    Loaded 109,851 rows
Loaded cached dim_org: 109,851 records


---
## 2. Define Chunking Strategy


In [4]:
# Get source tables for chunking
source_tables_query = """
    SELECT 
        source_table,
        COUNT(*) as record_count
    FROM allsci_prod_gold.potential_mismatched_organizations
    GROUP BY source_table
    ORDER BY record_count DESC
"""

source_tables_df = db.execute_query(source_tables_query, "Source table counts")
print("\nSOURCE TABLES FOR CHUNKING")
print("=" * 60)
print(source_tables_df.to_string())
print(f"\nTotal records: {source_tables_df['record_count'].sum():,}")


[15:23:17]  Executing: Source table counts
[15:23:20]    Returned 10 rows in 2.6s

SOURCE TABLES FOR CHUNKING
                                            source_table  record_count
0                              open_fda_silver.ndc_drugs      18471807
1                                   uspto_silver.patents      14849735
2      nih_clinical_trials_gov_silver.cl_trial_locations       4695492
3            who_clinical_trials_silver.studies_metadata       2884858
4  nih_clinical_trials_gov_silver.cl_trial_collaborators        620401
5  nih_clinical_trials_gov_silver.cl_trial_organizations        356592
6       nih_clinical_trials_gov_silver.cl_trial_sponsors        333591
7          chinese_clinical_trials_silver.trial_contacts        241022
8        legacy_alpha_silver._organizations_consolidated        140344
9                  chinese_clinical_trials_silver.trials        123630

Total records: 42,717,472


In [5]:
# Compute IDF scores from reference data
# These are used to extract distinctive tokens during inference
idf_scores = compute_token_statistics(dim_org_df, 'name_normalized')
corpus_stopwords = get_corpus_stopwords(idf_scores, percentile=0.10)

print(f"\nIDF SCORES COMPUTED")
print("=" * 50)
print(f"Vocabulary size: {len(idf_scores):,} tokens")
print(f"Corpus stopwords: {len(corpus_stopwords):,}")

[15:23:20]  Computing token IDF statistics...
[15:23:20]    Computed IDF for 64,820 unique tokens
[15:23:20]    IDF range: 1.90 (most common) to 11.61 (most rare)
[15:23:20]    Auto-identified 6786 corpus stopwords (IDF <= 10.00)

IDF SCORES COMPUTED
Vocabulary size: 64,820 tokens
Corpus stopwords: 6,786


In [6]:
# Configure chunking
# For demo, limit to first N records per source
# Set to None for full processing

DEMO_MODE = True  # Set to False for full processing
DEMO_LIMIT_PER_SOURCE = 1000 if DEMO_MODE else None

chunks = []
for _, row in source_tables_df.iterrows():
    chunks.append({
        'source_table': row['source_table'],
        'total_records': row['record_count'],
        'limit': DEMO_LIMIT_PER_SOURCE
    })

print(f"\nCHUNKING CONFIGURATION")
print("=" * 50)
print(f"Demo mode: {DEMO_MODE}")
print(f"Chunks: {len(chunks)}")
if DEMO_MODE:
    print(f"Limit per chunk: {DEMO_LIMIT_PER_SOURCE:,}")



CHUNKING CONFIGURATION
Demo mode: True
Chunks: 10
Limit per chunk: 1,000


---
## 3. Process Chunks


In [7]:
def process_chunk(chunk_info, dim_org_df, model_json, db, idf_scores, corpus_stopwords):
    """Process a single chunk of mismatched records"""
    source_table = chunk_info['source_table']
    limit = chunk_info['limit']
    
    log_step(f"Processing: {source_table}")
    
    # Load chunk
    chunk_df = load_mismatched(db, source_table=source_table, limit=limit)
    
    if len(chunk_df) == 0:
        log_step(f"  No records to process", "WARN")
        return pd.DataFrame()
    
    # Prepare data
    chunk_unified = create_unified_schema(chunk_df, 'mismatched')
    chunk_filtered, removed = filter_bad_records(chunk_unified)
    
    if len(chunk_filtered) == 0:
        log_step(f"  All records filtered out", "WARN")
        return pd.DataFrame()
    
    chunk_prepared = create_blocking_keys(chunk_filtered)
    
    # Add distinctive tokens using pre-computed IDF scores
    chunk_prepared = add_distinctive_tokens(chunk_prepared, idf_scores, corpus_stopwords)
    
    # Add token set features (name_tokens, all_names)
    chunk_prepared = add_token_set_features(chunk_prepared)
    
    log_step(f"  Prepared: {len(chunk_prepared):,} records (filtered {len(removed):,})")
    
    # Remove training metadata (not a Splink setting)
    model_settings = {k: v for k, v in model_json.items() if k != 'training_metadata'}
    
    # Initialize linker with model
    linker = Linker(
        [dim_org_df, chunk_prepared],
        model_settings,
        db_api=DuckDBAPI()
    )
    
    # Predict
    predictions = linker.inference.predict(
        threshold_match_probability=config.matching.THRESHOLD_PREDICTION
    )
    predictions_df = predictions.as_pandas_dataframe()
    
    log_step(f"  Predictions: {len(predictions_df):,}")
    
    predictions_df['source_table'] = source_table
    return predictions_df

In [8]:
# Load reference data (dim_org) for linking
cached_dim_org = load_checkpoint(config.paths.DATA_DIR + "/dim_org_training.parquet", "dim_org cache")

if cached_dim_org is not None:
    dim_org_df = cached_dim_org
    print(f"Loaded cached dim_org: {len(dim_org_df):,} records")
else:
    dim_org_raw = load_dim_org(db, sample_size=None)
    dim_org_df = create_unified_schema(dim_org_raw, 'dim_org')
    dim_org_df, _ = filter_bad_records(dim_org_df)
    dim_org_df = create_blocking_keys(dim_org_df)
    print(f"Loaded dim_org: {len(dim_org_df):,} records")

# Ensure dim_org has distinctive token features
if 'distinctive_token' not in dim_org_df.columns:
    print("Adding distinctive tokens to dim_org...")
    dim_org_df = add_distinctive_tokens(dim_org_df, idf_scores, corpus_stopwords)
    dim_org_df = add_token_set_features(dim_org_df)

[15:23:20]  Loading checkpoint: dim_org cache
[15:23:20]    Loaded 109,851 rows
Loaded cached dim_org: 109,851 records
Adding distinctive tokens to dim_org...
[15:23:20]  Adding distinctive tokens...
[15:23:20]    Loaded 34,655 geographic stopwords
[15:23:20]    Geographic stopwords: 34,655 (locations filtered out)
[15:23:21]    Distinctive token coverage: 99.9%
[15:23:21]  Adding token set features for containment detection...
[15:23:21]    Average token count: 3.2


In [9]:
# Process all chunks
all_predictions = []
tracker = ProgressTracker(len(chunks), "Inference Pipeline")

for i, chunk_info in enumerate(chunks):
    tracker.step(f"{chunk_info['source_table']} ({chunk_info['total_records']:,} total)")
    
    try:
        chunk_predictions = process_chunk(chunk_info, dim_org_df, model_json, db, idf_scores, corpus_stopwords)
        if len(chunk_predictions) > 0:
            all_predictions.append(chunk_predictions)
        
        # Running statistics
        total_so_far = sum(len(p) for p in all_predictions)
        log_step(f"  Running total: {total_so_far:,} predictions")
        log_step(f"  ETA: {tracker.eta()}")
        
    except Exception as e:
        log_step(f"  Error: {str(e)}", "ERROR")
        continue

tracker.complete()

[15:23:21]  [1/10] (10%) open_fda_silver.ndc_drugs (18,471,807 total)
[15:23:21]  Processing: open_fda_silver.ndc_drugs
[15:23:21]  Starting: Loading potential_mismatched_organizations
[15:23:21]  Executing: mismatched (source=open_fda_silver.ndc_drugs, limit=1000)
[15:23:25]    Returned 1,000 rows in 3.6s
[15:23:25]  Parsing metadata...
[15:23:25]    Extracted 1,000 records with metadata fields
[15:23:25]  Completed: Loading potential_mismatched_organizations (3.6s)
[15:23:25]  Creating unified schema for mismatched...
[15:23:25]    Created unified schema: 1,000 rows, 16 columns
[15:23:25]  Filtering bad records...
[15:23:25]    Removed 0 of 1,000 records (0.0%)
[15:23:25]  Creating blocking keys...
[15:23:25]  Adding phonetic codes...
[15:23:25]    Phonetic coverage: 100.0%
[15:23:25]    Added blocking keys to 1,000 records
[15:23:25]  Adding distinctive tokens...
[15:23:25]    Geographic stopwords: 34,655 (locations filtered out)
[15:23:25]    Distinctive token coverage: 100.0%
[15:

Blocking time: 0.02 seconds
Predict time: 0.14 seconds


[15:23:26]    Predictions: 373
[15:23:26]    Running total: 373 predictions
[15:23:26]    ETA: 43s
[15:23:26]  [2/10] (20%) uspto_silver.patents (14,849,735 total)
[15:23:26]  Processing: uspto_silver.patents
[15:23:26]  Starting: Loading potential_mismatched_organizations
[15:23:26]  Executing: mismatched (source=uspto_silver.patents, limit=1000)
[15:23:29]    Returned 1,000 rows in 3.5s
[15:23:29]  Parsing metadata...
[15:23:29]    Extracted 1,000 records with metadata fields
[15:23:29]  Completed: Loading potential_mismatched_organizations (3.6s)
[15:23:29]  Creating unified schema for mismatched...
[15:23:29]    Created unified schema: 1,000 rows, 16 columns
[15:23:29]  Filtering bad records...
[15:23:29]    Removed 6 of 1,000 records (0.6%)
[15:23:29]      - short_name: 6
[15:23:29]  Creating blocking keys...
[15:23:29]  Adding phonetic codes...
[15:23:29]    Phonetic coverage: 100.0%
[15:23:29]    Added blocking keys to 994 records
[15:23:29]  Adding distinctive tokens...
[15:23:

Blocking time: 0.02 seconds
Predict time: 0.14 seconds


[15:23:30]    Predictions: 418
[15:23:30]    Running total: 791 predictions
[15:23:30]    ETA: 38s
[15:23:30]  [3/10] (30%) nih_clinical_trials_gov_silver.cl_trial_locations (4,695,492 total)
[15:23:30]  Processing: nih_clinical_trials_gov_silver.cl_trial_locations
[15:23:30]  Starting: Loading potential_mismatched_organizations
[15:23:30]  Executing: mismatched (source=nih_clinical_trials_gov_silver.cl_trial_locations, limit=1000)
[15:23:33]    Returned 1,000 rows in 2.5s
[15:23:33]  Parsing metadata...
[15:23:33]    Extracted 1,000 records with metadata fields
[15:23:33]  Completed: Loading potential_mismatched_organizations (2.6s)
[15:23:33]  Creating unified schema for mismatched...
[15:23:33]    Created unified schema: 1,000 rows, 16 columns
[15:23:33]  Filtering bad records...
[15:23:33]    Removed 11 of 1,000 records (1.1%)
[15:23:33]      - generic_name: 11
[15:23:33]  Creating blocking keys...
[15:23:33]  Adding phonetic codes...
[15:23:33]    Phonetic coverage: 100.0%
[15:23:

...

[15:23:57]    Predictions: 24
[15:23:57]    Running total: 9,598 predictions
[15:23:57]    ETA: 9s
[15:23:57]  [9/10] (90%) legacy_alpha_silver._organizations_consolidated (140,344 total)
[15:23:57]  Processing: legacy_alpha_silver._organizations_consolidated
[15:23:57]  Starting: Loading potential_mismatched_organizations
[15:23:57]  Executing: mismatched (source=legacy_alpha_silver._organizations_consolidated, limit=1000)
[15:24:01]    Returned 1,000 rows in 3.6s
[15:24:01]  Parsing metadata...
[15:24:01]    Extracted 1,000 records with metadata fields
[15:24:01]  Completed: Loading potential_mismatched_organizations (3.6s)
[15:24:01]  Creating unified schema for mismatched...
[15:24:01]    Created unified schema: 1,000 rows, 16 columns
[15:24:01]  Filtering bad records...
[15:24:01]    Removed 1 of 1,000 records (0.1%)
[15:24:01]      - short_name: 1
[15:24:01]  Creating blocking keys...
[15:24:01]  Adding phonetic codes...
[15:24:01]    Phonetic coverage: 100.0%
[15:24:01]    Added

Blocking time: 0.02 seconds
Predict time: 0.14 seconds


[15:24:02]    Predictions: 40
[15:24:02]    Running total: 9,638 predictions
[15:24:02]    ETA: 5s
[15:24:02]  [10/10] (100%) chinese_clinical_trials_silver.trials (123,630 total)
[15:24:02]  Processing: chinese_clinical_trials_silver.trials
[15:24:02]  Starting: Loading potential_mismatched_organizations
[15:24:02]  Executing: mismatched (source=chinese_clinical_trials_silver.trials, limit=1000)
[15:24:05]    Returned 1,000 rows in 3.5s
[15:24:05]  Parsing metadata...
[15:24:05]    Extracted 1,000 records with metadata fields
[15:24:05]  Completed: Loading potential_mismatched_organizations (3.5s)
[15:24:05]  Creating unified schema for mismatched...
[15:24:05]    Created unified schema: 1,000 rows, 16 columns
[15:24:05]  Filtering bad records...
[15:24:05]    Removed 2 of 1,000 records (0.2%)
[15:24:05]      - short_name: 2
[15:24:05]  Creating blocking keys...
[15:24:05]  Adding phonetic codes...
[15:24:05]    Phonetic coverage: 100.0%
[15:24:05]    Added blocking keys to 998 record

Blocking time: 0.02 seconds
Predict time: 0.14 seconds


[15:24:07]    Predictions: 23
[15:24:07]    Running total: 9,661 predictions
[15:24:07]    ETA: 0s
[15:24:07]  Inference Pipeline complete in 45.6s


In [10]:
sample_who = load_mismatched(db, source_table='who_clinical_trials_silver.studies_metadata', limit=100)
print(sample_who.columns.tolist())
multi = sample_who[sample_who['country'].str.contains(';', na=False)]
print(f"Records with multi-country: {len(multi)}")
if len(multi) > 0:
    print(multi['country'].value_counts().head(10))

[15:24:07]  Starting: Loading potential_mismatched_organizations
[15:24:07]  Executing: mismatched (source=who_clinical_trials_silver.studies_metadata, limit=100)
[15:24:09]    Returned 100 rows in 2.4s
[15:24:09]  Parsing metadata...
[15:24:09]    Extracted 100 records with metadata fields
[15:24:09]  Completed: Loading potential_mismatched_organizations (2.5s)
['mismatch_id', 'name', 'source_table', 'source_entity_id', 'name_clean', 'name_normalized', 'name_prefix_5', 'name_prefix_10', 'type', 'country', 'city', 'state', 'latitude', 'longitude', 'name_aliases', 'rp_type', 'raw_metadata', 'source']
Records with multi-country: 9
country
United States;Canada;United States                                                                                                                                                                                             1
United States;Australia;Canada;New Zealand;Australia;Canada;New Zealand;United States                                             

In [11]:
multi

,mismatch_id,name,source_table,source_entity_id,name_clean,name_normalized,name_prefix_5,name_prefix_10,type,country,city,state,latitude,longitude,name_aliases,rp_type,raw_metadata,source
12,684fbe2b1415c2eb6530ce21a51a9319,Organon and Co,who_clinical_trials_silver.studies_metadata,NCT05172726,Organon and Co,organon and co,organ,organon an,None,United States;Canada;United States,None,None,None,None,None,None,"{name_clean=Organon and Co, contact_affiliation=Organon and Co, countries=Un...",mismatched
15,6547bda605362091bb44820078b03c41,Edwards Lifesciences,who_clinical_trials_silver.studies_metadata,NCT05172960,Edwards Lifesciences,edwards lifesciences,edwar,edwards li,None,United States;Australia;Canada;New Zealand;Australia;Canada;New Zealand;Unit...,None,None,None,None,None,None,"{name_clean=Edwards Lifesciences, contact_affiliation=Columbia University;St...",mismatched
33,711bc8d43270559c8a53e938186ece55,"Agios Pharmaceuticals, Inc.",who_clinical_trials_silver.studies_metadata,NCT05175105,"Agios Pharmaceuticals, Inc.",agios pharmaceuticals,agios,agios phar,None,United States;Canada;France;Germany;Netherlands;Spain;Switzerland;Canada;Fra...,None,None,None,None,None,None,"{name_clean=Agios Pharmaceuticals, Inc., contact_affiliation=Agios Pharmaceu...",mismatched
52,cd7163712ab217d4f3c1c66a5b771f61,Vitadx,who_clinical_trials_silver.studies_metadata,NCT05176145,Vitadx,vitadx,vitad,vitadx,None,Belgium;France;Spain;Belgium;France;Spain,None,None,None,None,None,None,"{name_clean=Vitadx, contact_affiliation=null, countries=Belgium;France;Spain...",mismatched
58,4bfcaefb6b6bb2d43961d46d2800eaf2,ProQR Therapeutics,who_clinical_trials_silver.studies_metadata,NCT05176717,ProQR Therapeutics,proqr therapeutics,proqr,proqr ther,None,United States;United Kingdom;United States,None,None,None,None,None,None,"{name_clean=ProQR Therapeutics, contact_affiliation=ProQR Therapeutics;ProQR...",mismatched
79,1f9383dcb78a158fd3fc8fc61bc07f74,"Scynexis, Inc.",who_clinical_trials_silver.studies_metadata,NCT05178862,"Scynexis, Inc.",scynexis,scyne,scynexis,None,United States;Belgium;Bulgaria;Canada;China;France;Germany;Greece;Israel;Ita...,None,None,None,None,None,None,"{name_clean=Scynexis, Inc., contact_affiliation=Scynexis, Inc., countries=Un...",mismatched
91,eed4e7c7e3032ae94277379afceab659,Canadian Cancer Trials Group,who_clinical_trials_silver.studies_metadata,NCT05180097,Canadian Cancer Trials Group,canadian cancer trials group,canad,canadian c,None,Australia;Canada;Australia;Canada,None,None,None,None,None,None,"{name_clean=Canadian Cancer Trials Group, contact_affiliation=BCCA-Vancouver...",mismatched
93,d78b06cff66426c17a30d6e94e72bbdc,Region Örebro County,who_clinical_trials_silver.studies_metadata,NCT05180175,Region Örebro County,region rebro county,regio,region öre,None,Denmark;Iceland;Norway;Sweden;Denmark;Iceland;Norway;Sweden,None,None,None,None,None,None,"{name_clean=Region Örebro County, contact_affiliation=null, countries=Denmar...",mismatched
96,432827cb50fac48b8836f090c2fd220c,Reistone Biopharma Company Limited,who_clinical_trials_silver.studies_metadata,NCT05181137,Reistone Biopharma Company Limited,reistone biopharma company,reist,reistone b,None,United States;China;Georgia;Poland;Ukraine;China;Georgia;Poland;Ukraine;Unit...,None,None,None,None,None,None,"{name_clean=Reistone Biopharma Company Limited, contact_affiliation=Reistone...",mismatched


In [12]:
# Combine all predictions
if all_predictions:
    combined_predictions = pd.concat(all_predictions, ignore_index=True)
    print(f"\nCOMBINED PREDICTIONS")
    print("=" * 50)
    print(f"Total predictions: {len(combined_predictions):,}")
    print(f"\nPredictions by source:")
    print(combined_predictions['source_table'].value_counts()) 
else:
    combined_predictions = pd.DataFrame()
    print("No predictions generated")



COMBINED PREDICTIONS
Total predictions: 9,661

Predictions by source:
source_table
nih_clinical_trials_gov_silver.cl_trial_organizations    3358
nih_clinical_trials_gov_silver.cl_trial_sponsors         1482
nih_clinical_trials_gov_silver.cl_trial_collaborators    1388
who_clinical_trials_silver.studies_metadata              1348
nih_clinical_trials_gov_silver.cl_trial_locations        1207
uspto_silver.patents                                      418
open_fda_silver.ndc_drugs                                 373
legacy_alpha_silver._organizations_consolidated            40
chinese_clinical_trials_silver.trial_contacts              24
chinese_clinical_trials_silver.trials                      23
Name: count, dtype: int64


In [13]:
# Disambiguate many-to-many matches
# Keep only the best match per mismatched record (unique_id_r)

print("DISAMBIGUATING MANY-TO-MANY MATCHES")
print("=" * 50)

# Count duplicates before disambiguation
dupes_per_mismatched = combined_predictions.groupby('unique_id_r').size()
multi_match_count = (dupes_per_mismatched > 1).sum()
print(f"Mismatched records with multiple matches: {multi_match_count:,}")

if multi_match_count > 0:
    # Tie-breaking criteria (in order of priority):
    # 1. Highest match_probability
    # 2. Country match (if country_code_r is available)
    # 3. Prefer shorter name (HQ entity over subsidiary with location suffix)
    # 4. Alphabetical (deterministic fallback)

    def add_tiebreaker_score(df):
        df = df.copy()
        
        # Country match bonus (if country available in mismatched)
        if 'country_code_r' in df.columns:
            df['country_bonus'] = (df['country_code_l'] == df['country_code_r']).astype(int)
        else:
            df['country_bonus'] = 0
        
        # Prefer shorter names (less specific = likely HQ)
        df['name_len'] = df['name_l'].str.len() if 'name_l' in df.columns else df['name_normalized_l'].str.len()
        
        return df

    combined_predictions = add_tiebreaker_score(combined_predictions)

    # Sort by tiebreakers, then keep first (best) per mismatched record
    combined_predictions_sorted = combined_predictions.sort_values(
        by=['unique_id_r', 'match_probability', 'country_bonus', 'name_len'],
        ascending=[True, False, False, True]
    )

    # Keep best match per mismatched record
    disambiguated = combined_predictions_sorted.groupby('unique_id_r').first().reset_index()

    print(f"\nBefore disambiguation: {len(combined_predictions):,} predictions")
    print(f"After disambiguation:  {len(disambiguated):,} predictions")
    print(f"Removed duplicates:    {len(combined_predictions) - len(disambiguated):,}")

    # Save full version before disambiguation (for debugging)
    save_checkpoint(combined_predictions, config.paths.DATA_DIR + "/predictions_all_matches.parquet", "All matches (before disambiguation)")
    
    # Use disambiguated version going forward
    combined_predictions = disambiguated

    # Clean up temp columns
    combined_predictions = combined_predictions.drop(columns=['country_bonus', 'name_len'], errors='ignore')
    
    print("\nDisambiguation complete - keeping best match per mismatched record")
else:
    print("No duplicate matches found - no disambiguation needed")


DISAMBIGUATING MANY-TO-MANY MATCHES
Mismatched records with multiple matches: 872

Before disambiguation: 9,661 predictions
After disambiguation:  2,064 predictions
Removed duplicates:    7,597
[15:24:09]  Saving checkpoint: All matches (before disambiguation)
[15:24:09]    Saved 9,661 rows to /Users/robertlalani/Desktop/entity_resolution_12_18_25/01-05-26/data/predictions_all_matches.parquet

Disambiguation complete - keeping best match per mismatched record


In [ ]:
# ============================================================================
# LLM VALIDATION - CRITICAL FIX
# ============================================================================
# This section was missing from the original pipeline!
# LLM judge was implemented but never called, resulting in 100% errors.

if config.llm_judge.ENABLE_LLM_VALIDATION:
    from anthropic import Anthropic
    from llm_judge import judge_match
    import os
    
    # Initialize Anthropic client
    api_key = os.environ.get("ANTHROPIC_API_KEY")
    if not api_key:
        log_step("WARNING: ANTHROPIC_API_KEY not set - skipping LLM validation", "WARN")
        log_step("Set with: export ANTHROPIC_API_KEY='your-key'", "WARN")
    else:
        client = Anthropic(api_key=api_key)
        
        print("\n" + "=" * 70)
        print("LLM VALIDATION PIPELINE")
        print("=" * 70)
        
        # ========================================================================
        # OPTION 1: MULTI-AGENT ROUTING (RECOMMENDED - 60% cost reduction)
        # ========================================================================
        if config.multi_agent.ENABLE_MULTI_AGENT:
            from multi_agent_validator import batch_validate_with_agents
            
            log_step("Using multi-agent routing for cost-efficient validation...")
            print(f"  Config: {config.llm_judge.PRIMARY_MODEL}")
            print(f"  Multi-agent routing: Enabled")
            print(f"  Predictions to validate: {len(combined_predictions):,}")
            
            # Apply active learning first if enabled
            if config.active_learning.ENABLE_ACTIVE_LEARNING:
                from active_learning import select_active_learning_batch
                
                log_step(f"Active learning: Selecting {config.active_learning.VALIDATION_BUDGET} highest-value samples...")
                
                validation_batch = select_active_learning_batch(
                    combined_predictions,
                    clusters_df,
                    budget=config.active_learning.VALIDATION_BUDGET,
                    min_per_source=config.active_learning.MIN_SAMPLES_PER_SOURCE,
                    source_weights=config.active_learning.SOURCE_WEIGHTS
                )
                
                print(f"  Selected {len(validation_batch):,} samples for validation")
                print(f"  Cost savings: {100 * (1 - len(validation_batch)/len(combined_predictions)):.1f}%")
            else:
                validation_batch = combined_predictions
                log_step("Validating all predictions (no active learning)")
            
            # Multi-agent validation
            combined_predictions, agent_stats = batch_validate_with_agents(
                validation_batch,
                dim_org_df,
                client,
                llm_judge_func=judge_match,
                enable_routing=True,
                primary_model=config.llm_judge.PRIMARY_MODEL,
                fast_model=config.llm_judge.FAST_MODEL
            )
            
            print("\nMULTI-AGENT STATISTICS:")
            print("=" * 70)
            print(f"Total validations: {agent_stats['total']:,}")
            if 'summary' in agent_stats:
                summary = agent_stats['summary']
                print(f"  Free (deterministic):  {summary['free_deterministic']:,} ({summary['cost_reduction_pct']:.1f}% savings)")
                print(f"  LLM calls required:    {summary['llm_required']:,}")
        
        # ========================================================================
        # OPTION 2: ACTIVE LEARNING ONLY (No multi-agent)
        # ========================================================================
        elif config.active_learning.ENABLE_ACTIVE_LEARNING:
            from active_learning import select_active_learning_batch
            from llm_judge import batch_judge_matches
            
            log_step(f"Using active learning: selecting {config.active_learning.VALIDATION_BUDGET} samples...")
            
            # Select high-value samples
            selected_batch = select_active_learning_batch(
                combined_predictions,
                clusters_df,
                budget=config.active_learning.VALIDATION_BUDGET,
                min_per_source=config.active_learning.MIN_SAMPLES_PER_SOURCE
            )
            
            print(f"  Selected: {len(selected_batch):,}/{len(combined_predictions):,} predictions")
            print(f"  Cost savings: {100 * (1 - len(selected_batch)/len(combined_predictions)):.1f}%")
            
            # Validate with LLM
            log_step(f"Validating {len(selected_batch):,} predictions with LLM...")
            
            llm_results = batch_judge_matches(
                selected_batch,
                dim_org_df,
                client,
                model=config.llm_judge.PRIMARY_MODEL,
                max_workers=config.llm_judge.MAX_WORKERS,
                enable_bias_mitigation=config.llm_judge.ENABLE_POSITION_BIAS_MITIGATION,
                enable_ensemble=config.llm_judge.ENABLE_ENSEMBLE,
                progress_callback=lambda i, total: print(f"  Progress: {i}/{total}", end='\r') if i % 50 == 0 else None
            )
            
            # Merge results back
            for result in llm_results:
                idx = result['prediction_idx']
                combined_predictions.loc[idx, 'llm_match'] = result['llm_match']
                combined_predictions.loc[idx, 'llm_confidence'] = result['llm_confidence']
                combined_predictions.loc[idx, 'llm_reason'] = result['llm_reason']
                if 'position_bias_detected' in result:
                    combined_predictions.loc[idx, 'position_bias_detected'] = result['position_bias_detected']
            
            log_step(f"LLM validation complete: {len(llm_results):,} results")
        
        # ========================================================================
        # OPTION 3: VALIDATE ALL (Most expensive)
        # ========================================================================
        else:
            from llm_judge import batch_judge_matches
            
            log_step(f"Validating ALL {len(combined_predictions):,} predictions with LLM...")
            log_step("WARNING: This is expensive! Consider enabling active_learning", "WARN")
            
            llm_results = batch_judge_matches(
                combined_predictions,
                dim_org_df,
                client,
                model=config.llm_judge.PRIMARY_MODEL,
                max_workers=config.llm_judge.MAX_WORKERS,
                enable_bias_mitigation=config.llm_judge.ENABLE_POSITION_BIAS_MITIGATION,
                enable_ensemble=config.llm_judge.ENABLE_ENSEMBLE,
                progress_callback=lambda i, total: print(f"  Progress: {i}/{total}", end='\r') if i % 100 == 0 else None
            )
            
            # Merge results
            for result in llm_results:
                idx = result['prediction_idx']
                combined_predictions.loc[idx, 'llm_match'] = result['llm_match']
                combined_predictions.loc[idx, 'llm_confidence'] = result['llm_confidence']
                combined_predictions.loc[idx, 'llm_reason'] = result['llm_reason']
            
            log_step(f"LLM validation complete: {len(llm_results):,} results")
        
        # ========================================================================
        # CONFIDENCE CALIBRATION (if enabled)
        # ========================================================================
        if config.llm_judge.ENABLE_CALIBRATION and 'llm_confidence' in combined_predictions.columns:
            from llm_judge import calibrate_confidence
            
            log_step("Calibrating LLM confidence scores...")
            
            try:
                calibrated = calibrate_confidence(combined_predictions)
                combined_predictions['llm_confidence_calibrated'] = calibrated
                log_step("Calibration complete")
            except Exception as e:
                log_step(f"Calibration failed: {str(e)}", "WARN")
        
        # ========================================================================
        # VALIDATION SUMMARY
        # ========================================================================
        print("\n" + "=" * 70)
        print("LLM VALIDATION SUMMARY")
        print("=" * 70)
        
        if 'llm_match' in combined_predictions.columns:
            validated = combined_predictions['llm_match'].notna()
            total_validated = validated.sum()
            confirmed = (combined_predictions['llm_match'] == True).sum()
            rejected = (combined_predictions['llm_match'] == False).sum()
            errors = (combined_predictions['llm_match'].isna()).sum()
            
            print(f"\nValidated: {total_validated:,}/{len(combined_predictions):,} predictions")
            print(f"  Confirmed matches: {confirmed:,} ({100*confirmed/total_validated:.1f}%)")
            print(f"  Rejected matches:  {rejected:,} ({100*rejected/total_validated:.1f}%)")
            if errors > 0:
                print(f"  Errors:            {errors:,}")
            
            # Disagreements
            if confirmed + rejected > 0:
                high_splink_rejected = combined_predictions[
                    (combined_predictions['match_probability'] >= 0.95) &
                    (combined_predictions['llm_match'] == False)
                ]
                
                low_splink_confirmed = combined_predictions[
                    (combined_predictions['match_probability'] < 0.70) &
                    (combined_predictions['llm_match'] == True)
                ]
                
                print(f"\nDisagreements:")
                print(f"  High Splink (>=0.95) but LLM rejected: {len(high_splink_rejected):,}")
                print(f"  Low Splink (<0.70) but LLM confirmed:  {len(low_splink_confirmed):,}")
                
                if len(high_splink_rejected) > 0 or len(low_splink_confirmed) > 0:
                    print(f"\n  → Consider reviewing disagreements for feedback!")
        
        print("\n" + "=" * 70)
        log_step("LLM validation pipeline complete!")

else:
    log_step("LLM validation disabled in config", "WARN")
    print("  To enable: config.llm_judge.ENABLE_LLM_VALIDATION = True")

---
## 3.5. LLM Validation with Multi-Agent Routing

CRITICAL FIX: This cell was missing - LLM validation was configured but never executed!

Now using multi-agent routing for cost-efficient validation:
- Active learning: Select highest-value samples (75% cost reduction)
- Multi-agent: Route to specialized agents (60% fewer LLM calls)
- Bias mitigation: Position bias detection (optional)
- Calibration: Confidence score calibration

In [14]:
# Cluster predictions at threshold using graph-based approach
if len(combined_predictions) > 0:
    print("CLUSTERING PREDICTIONS")
    print("=" * 50)
    
    import networkx as nx
    
    # Filter to threshold
    high_conf = combined_predictions[
        combined_predictions['match_probability'] >= config.matching.CLUSTER_THRESHOLD
    ]
    
    print(f"Building graph from {len(high_conf):,} edges above threshold {config.matching.CLUSTER_THRESHOLD}")
    
    # Build graph from pairwise predictions
    G = nx.Graph()
    
    # Add edges for each prediction above threshold
    for _, row in high_conf.iterrows():
        G.add_edge(row['unique_id_l'], row['unique_id_r'], 
                   weight=row['match_probability'])
    
    # Find connected components (clusters)
    components = list(nx.connected_components(G))
    
    # Create clusters dataframe
    cluster_records = []
    for cluster_id, members in enumerate(components):
        for unique_id in members:
            cluster_records.append({
                'unique_id': unique_id,
                'cluster_id': cluster_id
            })
    
    clusters_df = pd.DataFrame(cluster_records)
    
    # Cluster statistics
    cluster_sizes = clusters_df.groupby('cluster_id').size()
    singleton_count = (cluster_sizes == 1).sum()
    multi_count = (cluster_sizes > 1).sum()
    
    print(f"\nClustering threshold: {config.matching.CLUSTER_THRESHOLD}")
    print(f"Total clusters: {len(cluster_sizes):,}")
    print(f"  Singleton clusters (1 record):  {singleton_count:,}")
    print(f"  Multi-record clusters (2+):     {multi_count:,}")
    print(f"  Largest cluster:                {cluster_sizes.max()} records")
    
    # Merge cluster_id back to predictions
    combined_predictions = combined_predictions.merge(
        clusters_df.rename(columns={'unique_id': 'unique_id_l'}),
        on='unique_id_l',
        how='left'
    )
    
    # Save clusters
    save_checkpoint(clusters_df, config.paths.CLUSTERS, "Entity clusters")
    print(f"\nClusters saved to: {config.paths.CLUSTERS}")
else:
    print("No predictions to cluster")

CLUSTERING PREDICTIONS
Building graph from 2,064 edges above threshold 0.85

Clustering threshold: 0.85
Total clusters: 712
  Singleton clusters (1 record):  0
  Multi-record clusters (2+):     712
  Largest cluster:                143 records
[15:24:09]  Saving checkpoint: Entity clusters
[15:24:09]    Saved 2,776 rows to /Users/robertlalani/Desktop/entity_resolution_12_18_25/01-05-26/data/clusters.parquet

Clusters saved to: /Users/robertlalani/Desktop/entity_resolution_12_18_25/01-05-26/data/clusters.parquet


---
## 4. Prediction Statistics


In [15]:
# Prediction score distribution
if len(combined_predictions) > 0:
    print("PREDICTION SCORE DISTRIBUTION")
    print("=" * 50)
    
    bins = [0, 0.5, 0.7, 0.85, 0.95, 1.0]
    labels = ['0.5-0.7 (Low)', '0.7-0.85 (Medium)', '0.85-0.95 (High)', '0.95-1.0 (Very High)']
    combined_predictions['confidence_tier'] = pd.cut(
        combined_predictions['match_probability'], 
        bins=bins[1:], 
        labels=labels
    )
    
    print("\nConfidence tiers:")
    tier_counts = combined_predictions['confidence_tier'].value_counts().sort_index()
    for tier, count in tier_counts.items():
        pct = 100 * count / len(combined_predictions)
        print(f"  {tier:<30} | {count:>8,} ({pct:5.1f}%)")


PREDICTION SCORE DISTRIBUTION

Confidence tiers:
  0.5-0.7 (Low)                  |        0 (  0.0%)
  0.7-0.85 (Medium)              |        0 (  0.0%)
  0.85-0.95 (High)               |       12 (  0.6%)
  0.95-1.0 (Very High)           |    2,052 ( 99.4%)


In [16]:
combined_predictions.to_csv('predictions.csv')

In [ ]:
import os



---
## 5. LLM Validation (Optional)

Use Claude to validate predictions with source-aware context.


In [ ]:
# LLM Validation Configuration
import os

ENABLE_LLM_VALIDATION = True  # Set to False to skip LLM validation
LLM_SAMPLE_SIZE = None  # None = all predictions, or int for sample
LLM_MODEL = "claude-sonnet-4-20250514"  # or "claude-3-5-haiku-20241022" for cheaper

# Set your Anthropic API key (uncomment one option):
# Option 1: Set directly (not recommended for shared notebooks)

# Option 2: Load from .env file or shell environment (recommended)

# Check if API key is set





LLM Validation: Enabled
API Key: Set
Sample size: All predictions
Model: claude-sonnet-4-20250514


In [27]:
pip install anthropic

Note: you may need to restart the kernel to use updated packages.


In [25]:
combined_predictions

,unique_id_r,match_weight,match_probability,source_dataset_l,source_dataset_r,unique_id_l,name_normalized_l,name_normalized_r,gamma_name_normalized,distinctive_soundex_l,distinctive_soundex_r,gamma_phonetic_match,distinctive_tokens_l,distinctive_tokens_r,gamma_distinctive_match,gamma_token_overlap,name_tokens_l,name_tokens_r,gamma_containment_check,distinctive_token_l,distinctive_token_r,gamma_first_token_match,all_names_l,all_names_r,name_l,name_r,gamma_alias_match,country_code_l,country_code_r,gamma_country_match,city_l,city_r,gamma_city_match,name_metaphone_l,name_metaphone_r,match_key,source_table,cluster_id,confidence_tier,llm_match,llm_confidence,llm_reason,agreement
0,mis_002cf792e88bbb56fb08e8f0c8af0f59,37.454244,1.000000,__splink__input_table_0,__splink__input_table_1,dim_ASC-OR-0000000041087-1.0-1724880247,alvogen,alvogen,4,A412,A412,1,[alvogen],[alvogen],1,1,[alvogen],[alvogen],4,alvogen,alvogen,3,"[Alvogen (South Korea), [Alvogen (South Korea)]]",[Alvogen Inc.],Alvogen (South Korea),Alvogen Inc.,0,KR,None,-1,Seoul,None,-1,ALFJN,ALFJN,0,open_fda_silver.ndc_drugs,0,0.95-1.0 (Very High),None,0,"error: Error code: 404 - {'type': 'error', 'error': {'type': 'not_found_erro...",False
1,mis_00542559c67d52338487be0edf544732,46.968257,1.000000,__splink__input_table_0,__splink__input_table_1,dim_ASC-OR-0000000087218-1.0-1724880259,worldwide clinical trials,worldwide clinical trials,4,T642,T642,1,"[trials, clinical]","[trials, clinical]",2,2,"[clinical, trials, worldwide]","[clinical, trials, worldwide]",4,trials,trials,3,"[Worldwide Clinical Trials (United States), [Worldwide Clinical Trials (Unit...",[Worldwide Clinical Trials],Worldwide Clinical Trials (United States),Worldwide Clinical Trials,0,US,None,-1,Morrisville,None,-1,WRLTWT,WRLTWT,0,nih_clinical_trials_gov_silver.cl_trial_collaborators,1,0.95-1.0 (Very High),None,0,"error: Error code: 404 - {'type': 'error', 'error': {'type': 'not_found_erro...",False
2,mis_0055b6aece3ec9eb81d53455a3826c03,34.260333,1.000000,__splink__input_table_0,__splink__input_table_1,dim_ASC-OR-0000000060300-1.0-1724880251,aurora st lukes medical center,aurora saint lukes medical center,2,L220,L220,1,[lukes],"[lukes, saint]",1,1,"[aurora, center, lukes, medical, st]","[aurora, center, lukes, medical, saint]",3,lukes,lukes,3,"[Aurora St. Luke's Medical Center, [Aurora St. Luke's Medical Center]]",[Aurora Saint Luke's Medical Center],Aurora St. Luke's Medical Center,Aurora Saint Luke's Medical Center,0,US,US,1,Milwaukee,Milwaukee,2,ARR,ARR,2,nih_clinical_trials_gov_silver.cl_trial_locations,2,0.95-1.0 (Very High),None,0,"error: Error code: 404 - {'type': 'error', 'error': {'type': 'not_found_erro...",False
3,mis_0055df230e18cd791196945b144f2599,17.477389,0.999995,__splink__input_table_0,__splink__input_table_1,dim_ASC-OR-0000000096560-1.0-1724880262,kaiser permanente franklin medical offices,kaiser permanentefranklin,2,O122,P655,0,"[offices, permanente, kaiser]","[permanentefranklin, kaiser]",1,1,"[franklin, kaiser, medical, offices, permanente]","[kaiser, permanentefranklin]",2,offices,permanentefranklin,0,"[Kaiser Permanente Franklin Medical Offices, [Kaiser Permanente Franklin Med...",[Kaiser Permanente-Franklin],Kaiser Permanente Franklin Medical Offices,Kaiser Permanente-Franklin,0,US,US,1,Denver,Denver,2,KSR,KSR,3,nih_clinical_trials_gov_silver.cl_trial_locations,3,0.95-1.0 (Very High),None,0,"error: Error code: 404 - {'type': 'error', 'error': {'type': 'not_found_erro...",False
4,mis_00b350e6571b0d87b0de54b0e60dfead,36.071466,1.000000,__splink__input_table_0,__splink__input_table_1,dim_ASC-OR-0000000029265-1.0-1724880246,bayer,bayer,4,B600,B600,1,[bayer],[bayer],1,1,[bayer],[bayer],4,bayer,bayer,3,"[[Bayer (Italy)], Bayer (Italy)]",[Bayer],Bayer (Italy),Bayer,0,IT,TW,0,Milan,None,-1,BYR,BYR,0,who_clinical_trials_silver.studies_metadata,4,0.95-1.0 (Very High),None,0,"error: Error code: 404 - {'type': 'error', 'error': {'type': 'not_found_erro...",False
...,...,...,...,...,...,...,...,...,.

In [22]:
# LLM Validation Summary
if 'llm_match' in combined_predictions.columns:
    print("LLM VALIDATION SUMMARY")
    print("=" * 60)
    
    # Overall stats
    total = len(combined_predictions)
    llm_true = (combined_predictions['llm_match'] == True).sum()
    llm_false = (combined_predictions['llm_match'] == False).sum()
    llm_error = combined_predictions['llm_match'].isna().sum()
    
    print(f"\nOverall Results ({total:,} predictions):")
    print(f"  LLM Confirmed Match:    {llm_true:>6,} ({100*llm_true/total:5.1f}%)")
    print(f"  LLM Rejected:           {llm_false:>6,} ({100*llm_false/total:5.1f}%)")
    print(f"  LLM Errors:             {llm_error:>6,} ({100*llm_error/total:5.1f}%)")
    
    # By source table
    print("\nBy Source Table:")
    by_source = combined_predictions.groupby('source_table').agg({
        'llm_match': lambda x: (x == True).sum(),
        'match_probability': 'count'
    }).rename(columns={'llm_match': 'confirmed', 'match_probability': 'total'})
    by_source['rejected'] = by_source['total'] - by_source['confirmed']
    by_source['confirm_rate'] = 100 * by_source['confirmed'] / by_source['total']
    print(by_source.sort_values('total', ascending=False).to_string())
    
    # Disagreements (high Splink score but LLM rejected)
    disagreements = combined_predictions[
        (combined_predictions['match_probability'] >= 0.95) & 
        (combined_predictions['llm_match'] == False)
    ]
    print(f"\nDisagreements (Splink >= 0.95 but LLM rejected): {len(disagreements):,}")
    
    if len(disagreements) > 0:
        print("\nSample disagreements:")
        sample_cols = ['name_l', 'name_r', 'match_probability', 'llm_confidence', 'llm_reason', 'source_table']
        available_cols = [c for c in sample_cols if c in disagreements.columns]
        print(disagreements[available_cols].head(10).to_string())


LLM VALIDATION SUMMARY

Overall Results (2,064 predictions):
  LLM Confirmed Match:         0 (  0.0%)
  LLM Rejected:                0 (  0.0%)
  LLM Errors:              2,064 (100.0%)

By Source Table:
                                                       confirmed  total  rejected  confirm_rate
source_table                                                                                   
nih_clinical_trials_gov_silver.cl_trial_locations              0    465       465           0.0
nih_clinical_trials_gov_silver.cl_trial_organizations          0    419       419           0.0
nih_clinical_trials_gov_silver.cl_trial_collaborators          0    341       341           0.0
who_clinical_trials_silver.studies_metadata                    0    233       233           0.0
nih_clinical_trials_gov_silver.cl_trial_sponsors               0    201       201           0.0
open_fda_silver.ndc_drugs                                      0    195       195           0.0
uspto_silver.patents       

---
## 6. Export Results


In [23]:
# Save predictions
if len(combined_predictions) > 0:
    save_checkpoint(combined_predictions, config.paths.PREDICTIONS, "All predictions")
    
    # Save LLM-validated predictions separately if available
    if 'llm_match' in combined_predictions.columns:
        # Confirmed matches (LLM agrees)
        confirmed = combined_predictions[combined_predictions['llm_match'] == True]
        if len(confirmed) > 0:
            save_checkpoint(
                confirmed, 
                config.paths.DATA_DIR + "/llm_confirmed_matches.parquet", 
                "LLM confirmed matches"
            )
        
        # Rejected matches (LLM disagrees)
        rejected = combined_predictions[combined_predictions['llm_match'] == False]
        if len(rejected) > 0:
            save_checkpoint(
                rejected, 
                config.paths.DATA_DIR + "/llm_rejected_matches.parquet", 
                "LLM rejected matches"
            )
    
    # Save high-confidence matches separately
    high_conf = combined_predictions[combined_predictions['match_probability'] >= config.matching.THRESHOLD_HIGH_CONFIDENCE]
    if len(high_conf) > 0:
        save_checkpoint(high_conf, config.paths.DATA_DIR + "/high_confidence_matches.parquet", "High confidence matches")
    
    print(f"\nResults saved:")
    print(f"  - All predictions: {config.paths.PREDICTIONS}")
    print(f"  - High confidence: {config.paths.DATA_DIR}/high_confidence_matches.parquet")


[15:27:59]  Saving checkpoint: All predictions
[15:27:59]    Saved 2,064 rows to /Users/robertlalani/Desktop/entity_resolution_12_18_25/01-05-26/data/predictions.parquet
[15:27:59]  Saving checkpoint: High confidence matches
[15:27:59]    Saved 2,052 rows to /Users/robertlalani/Desktop/entity_resolution_12_18_25/01-05-26/data/high_confidence_matches.parquet

Results saved:
  - All predictions: /Users/robertlalani/Desktop/entity_resolution_12_18_25/01-05-26/data/predictions.parquet
  - High confidence: /Users/robertlalani/Desktop/entity_resolution_12_18_25/01-05-26/data/high_confidence_matches.parquet


In [24]:
# Summary
print("\n" + "=" * 70)
print("INFERENCE COMPLETE")
print("=" * 70)

if len(combined_predictions) > 0:
    llm_info = ""
    if 'llm_match' in combined_predictions.columns:
        confirmed = (combined_predictions['llm_match'] == True).sum()
        rejected = (combined_predictions['llm_match'] == False).sum()
        llm_info = f"""
LLM VALIDATION:
  - LLM Confirmed: {confirmed:,}
  - LLM Rejected: {rejected:,}
  - Confirm Rate: {100*confirmed/(confirmed+rejected):.1f}%"""
    
    print(f"""
RESULTS SUMMARY:
  - Total predictions: {len(combined_predictions):,}
  - High confidence (>0.95): {(combined_predictions['match_probability'] > 0.95).sum():,}
  - Sources processed: {len(chunks)}{llm_info}

OUTPUT FILES:
  - All predictions: {config.paths.PREDICTIONS}
  - High confidence: {config.paths.DATA_DIR}/high_confidence_matches.parquet""")
    
    if 'llm_match' in combined_predictions.columns:
        print(f"""  - LLM confirmed: {config.paths.DATA_DIR}/llm_confirmed_matches.parquet
  - LLM rejected: {config.paths.DATA_DIR}/llm_rejected_matches.parquet""")
    
    print("""
NEXT STEPS:
  1. Run 4_analysis.ipynb to review results
  2. Review LLM rejected matches for false positives
  3. Export final matches for production use
""")

# Cleanup
db.close()
print("Database connection closed")



INFERENCE COMPLETE

RESULTS SUMMARY:
  - Total predictions: 2,064
  - High confidence (>0.95): 2,052
  - Sources processed: 10
LLM VALIDATION:
  - LLM Confirmed: 0
  - LLM Rejected: 0
  - Confirm Rate: nan%

OUTPUT FILES:
  - All predictions: /Users/robertlalani/Desktop/entity_resolution_12_18_25/01-05-26/data/predictions.parquet
  - High confidence: /Users/robertlalani/Desktop/entity_resolution_12_18_25/01-05-26/data/high_confidence_matches.parquet
  - LLM confirmed: /Users/robertlalani/Desktop/entity_resolution_12_18_25/01-05-26/data/llm_confirmed_matches.parquet
  - LLM rejected: /Users/robertlalani/Desktop/entity_resolution_12_18_25/01-05-26/data/llm_rejected_matches.parquet

NEXT STEPS:
  1. Run 4_analysis.ipynb to review results
  2. Review LLM rejected matches for false positives
  3. Export final matches for production use

Database connection closed


/var/folders/ft/1328jrdx05s3bnd035tpxmb80000gn/T/ipykernel_20528/4211970717.py:15: RuntimeWarning: invalid value encountered in scalar divide
  - Confirm Rate: {100*confirmed/(confirmed+rejected):.1f}%"""
